# Benchmark a LOAM model from the Hugging Face Hub

This notebook runs a released LOAM model through the **three biological tasks of the LOAM
manuscript**, including the layer sweeps, and compares the result with the numbers reported
in the paper.

| Task row | What is measured | How the model is read |
|---|---|---|
| `bacbench-essentiality` | is a bacterial gene essential? (macro-mean AUROC over held-out genomes) | linear probe on mean-pooled embeddings of the **last** layer |
| `bacbench-essentiality-layer-sweep` | same, one probe per layer; reports the best layer | every layer, from the same forward pass |
| `dgeb-ec-classification-dna` | Enzyme Commission class from the gene's DNA (macro-F1, 128 classes) | linear probe, last layer |
| `dgeb-ec-classification-dna-layer-sweep` | same, per layer | every layer |
| `rnagym-dms` | rank single-nucleotide variants by measured fitness (mean \|Spearman\| over 11 assays) | zero-shot: mean log-likelihood of each variant sequence |

Nothing is fine-tuned. The best-layer value is chosen on the test split, so, as in the paper,
it is a retrospective upper bound; the last-layer value is reported next to it.

**Requirements**
- A Python environment with `glmbench` installed with its Hugging Face and data extras (`pip install -e ".[hf,data,notebook]"` from the repository root; see `docs/INSTALL.md`). Tested with torch 2.12 and transformers 5.12.1.
- Access to the model on the Hub: the `Soilytix/LOAM-*` repositories are gated. Accept the terms on the model page, then log in (`hf auth login`).
- A GPU is strongly recommended. Most of the time goes to embedding the essentiality corpus (164,630 windows); LOAM-25M is the quickest, and the larger models take proportionally longer.
- About 230 MB of task data, fetched below on first use.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

# --- what to run -----------------------------------------------------------------------
MODEL = "LOAM-25M"            # LOAM-25M | LOAM-100M | LOAM-340M | LOAM-624M
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # the repository root
# A Hub repo id, or a local directory holding a download of it. The benchmark hashes the
# weight bytes, so both give the same model hash.
MODEL_DIR = f"Soilytix/{MODEL}"
REVISION = None               # pin a Hub commit sha for exact reproducibility
GPU = "0"                     # physical GPU index; set to None to run on CPU (slow)
RESULTS_DIR = REPO / "results"

if GPU is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU
os.chdir(REPO)
print("repository:", REPO)
print("model:     ", MODEL_DIR)

## 1. The model is an ordinary Hugging Face model

LOAM ships its own modeling code, so it loads with `trust_remote_code=True`. The benchmark reads
it with the conventions of the paper's scoring code, which differ from the model card's
general-use examples (`docs/USING_LOAM.md` compares the two). The benchmark adapter applies
them for you:

- **A BOS token before the sequence**, as context only. LOAM was trained with no token before a
  sequence, so this changes the numbers slightly, but it is what the paper's scores used.
- **Context is 8,191 nt**, not 8,192, because BOS takes one of the 8,192 positions.
- **Embeddings are the raw output of each block.** `output_hidden_states=True` returns the
  *final-normed* state as its last element, and on LOAM that tensor is dominated by a few huge
  residual dimensions. The benchmark reads the blocks with forward hooks, as below.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() and GPU is not None else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, revision=REVISION, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, revision=REVISION, trust_remote_code=True).to(device).eval()
cfg = model.config
print(f"{MODEL}: {cfg.num_hidden_layers} blocks, width {cfg.hidden_size}, "
      f"{sum(p.numel() for p in model.parameters()):,} parameters, context {cfg.max_position_embeddings - 1} nt")

seq = "ATGAAAAAGCTGCTGGCAATTGCCGGTCTGGCCGCTGGCGCAGCCGCACAGGCAGAAGAAACCTGA"
ids = torch.tensor([[cfg.bos_token_id] + tokenizer(seq, add_special_tokens=False)["input_ids"]], device=device)

# Zero-shot score: mean log-probability of each nucleotide given BOS and the nucleotides before it.
with torch.inference_mode():
    logits = model(input_ids=ids).logits.float()
logp = torch.log_softmax(logits[0, :-1], dim=-1).gather(-1, ids[0, 1:, None]).squeeze(-1)
print(f"mean log-likelihood: {logp.mean().item():.4f} nats per nucleotide over {len(seq)} nt")

# Embeddings: the token-embedding output (layer 0) and every block's raw output (layers 1..L).
taps = []
hooks = [model.model.embed_tokens.register_forward_hook(lambda m, i, o: taps.append(o))]
hooks += [blk.register_forward_hook(lambda m, i, o: taps.append(o)) for blk in model.model.layers]
with torch.inference_mode():
    model.model(input_ids=ids)
for h in hooks:
    h.remove()
per_layer = torch.stack(taps)[:, 0, 1:].float().mean(dim=1)   # [layers, hidden]: drop BOS, mean over nucleotides
print("mean-pooled embedding per layer:", tuple(per_layer.shape))

del model
if device == "cuda":
    torch.cuda.empty_cache()

## 2. Task data

The task data is not redistributed here. The fetch scripts download it from its public
sources, rebuild the exact inputs the tasks use, and check every file against the sha256
pinned in the task's `manifest.yaml`. A mismatch is an error, never a silent substitution.

In [ ]:
DATA = REPO / "src" / "glmbench" / "tasks" / "data"
needed = {
    "scripts/fetch_bacbench_essentiality.py": [DATA / "bacbench_essentiality" / f"essential_genes_{s}.csv" for s in ("train", "test")],
    "scripts/fetch_dgeb_ec_dna.py": [DATA / "dgeb_ec_dna" / f"ec_classification_dna_{s}.csv" for s in ("train", "test")],
    "scripts/fetch_rnagym_prok.py": [DATA / "rnagym" / "processed_DMS_files"],
}
for script, paths in needed.items():
    present = all(p.exists() and (p.is_file() or any(p.iterdir())) for p in paths)
    print(f"{'present ' if present else 'fetching'}  {script}")
    if not present:
        subprocess.run([sys.executable, str(REPO / script)], check=True)

## 3. Run the benchmark

`loam_paper` is the benchmark definition with the five task rows above. The `loam-hf` adapter
runs the model in a subprocess (here, this same Python environment) and reproduces the
serving arithmetic of the implementation the paper was scored with: fp32 load then bf16 on
GPU, the same batching, mean pooling in bf16. That is why the numbers below can be compared
with the paper's to the last digit.

The essentiality probe and the sweep share one forward pass, so the sweep costs one extra
probe fit per layer and no extra GPU time. Results are written after every task to
`results/loam_paper/<model hash>.json`. Re-running this cell resumes: completed tasks are
skipped.

In [ ]:
import logging
from glmbench.config.benchmark_spec import resolve_benchmark_spec
from glmbench.config.model_spec import ModelSpec
from glmbench.run import run_benchmark

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

spec = ModelSpec(
    adapter="loam-hf",
    adapter_version="1.0.0",
    display_name=MODEL,
    model={"checkpoint": str(MODEL_DIR), "revision": REVISION},
    runner={"python_exe": sys.executable, "gpus": "cpu" if GPU is None else "0"},
)
benchmark = resolve_benchmark_spec("loam_paper")
record = run_benchmark(spec, benchmark, results_dir=RESULTS_DIR)
print("model hash:", record["model_hash"])

## 4. Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    display
except NameError:          # running as a plain script
    display = print

by_task = {t["task"]: t for t in record["tasks"]}
summary = pd.DataFrame([
    {"task": t["task"], "status": t["status"], "metric": t["primary_metric"],
     "value": t["metrics"].get(t["primary_metric"]), "best layer": t["metrics"].get("best_layer")}
    for t in record["tasks"]
])
display(summary)

# The model-free k-mer composition floor of each task (paper/kmer_floor.json).
floors = {k: v["floor"]["value"] for k, v in json.loads((REPO / "paper" / "kmer_floor.json").read_text())["tasks"].items()
          if v.get("floor")}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, task in zip(axes, ["bacbench-essentiality-layer-sweep", "dgeb-ec-classification-dna-layer-sweep"]):
    m, pm = by_task[task]["metrics"], by_task[task]["primary_metric"]
    layers = sorted(int(k.split("_")[1]) for k in m if k.startswith("layer_") and k.endswith("_" + pm))
    ax.plot(layers, [m[f"layer_{i}_{pm}"] for i in layers], marker="o", label=MODEL)
    if task in floors:
        ax.axhline(floors[task], ls="--", c="grey", label="k-mer composition floor")
    ax.set(title=task, xlabel="layer (0 = token embeddings)", ylabel=pm)
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

rm = by_task["rnagym-dms"]["metrics"]
display(pd.Series({k[: -len("_spearman")]: v for k, v in rm.items()
                   if k.endswith("_spearman") and k not in ("macro_spearman", "blat_ecolx_spearman")},
                  name="|Spearman|").sort_values(ascending=False).to_frame())

## 5. Compare with the paper

The paper's numbers ship with this repository in `paper/`: its records, and every metric of
every record in one table, `paper/reference_scores.csv`. The paper scored the same weights
through the native LOAM implementation; the released weights are those checkpoints, exported to
the Hugging Face format, and the adapter reproduces the native arithmetic. In the reference
environment (torch 2.12.0+cu126, transformers 5.12.1, `envs/constraints-paper.txt`)
the expected difference is therefore **exactly zero**. On other GPUs or library versions, bf16
kernels and BLAS builds can move the last digits.

In [ ]:
import csv

ref = [r for r in csv.DictReader(open(REPO / "paper" / "reference_scores.csv")) if r["model_id"] == MODEL]
headline = {r["task"]: float(r["value"]) for r in ref if r["is_primary"] == "1" and r["layer"] == ""}
best = {r["task"]: int(float(r["value"])) for r in ref if r["metric"] == "best_layer" and r["layer"] == ""}
rows = []
for task, paper_value in headline.items():
    m = by_task[task]["metrics"]
    ours = m[by_task[task]["primary_metric"]]
    rows.append({"task": task, "paper": paper_value, "this run": ours, "Δ": ours - paper_value,
                 "paper best layer": best.get(task), "this run best layer": m.get("best_layer")})
comparison = pd.DataFrame(rows)
display(comparison)
print("bitwise equal to the paper on all five headline values:" , bool((comparison["Δ"] == 0).all()))

`scripts/compare_to_paper.py` compares **every** metric: each tap of both sweeps, each
RNAGym assay, and the count-type metrics. Without `--exact` it applies per-task tolerances
meant for other hardware; add `--exact` to require bitwise agreement.

In [ ]:
cand_path = RESULTS_DIR / "loam_paper" / f"{record['model_hash'].replace(':', '_')}.json"
result = subprocess.run([sys.executable, str(REPO / "scripts" / "compare_to_paper.py"), str(cand_path), "--exact"],
                        capture_output=True, text=True)
print(result.stdout[-6000:])
print("bitwise reproduction of every metric:", result.returncode == 0)